# Week 55, Day 5: Regularised encoding models: estimating a receptive field

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w055/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Biased or noisy

Compute the spike-triggered average (rescaled by the mean pixel variance so its units match) and the least-squares estimate from 200 trials, and compare both with the true field by relative error and by width (the standard deviation of |k| across pixels).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(555)
P = 30
pix = np.arange(P)
k_true = np.exp(-((pix - 15) ** 2) / 30) * np.cos((pix - 15) * 0.6)

def stimuli(n):
    z = r.normal(size=(n, P + 4))
    return sum(z[:, i:i + P] for i in range(5)) / np.sqrt(5)

def responses(X):
    return X @ k_true + 2.0 * r.normal(size=len(X))

rel_err = lambda k: np.linalg.norm(k - k_true) / np.linalg.norm(k_true)
width = lambda k: np.sqrt(np.sum(np.abs(k) * (pix - np.sum(np.abs(k) * pix) / np.abs(k).sum()) ** 2) / np.abs(k).sum())

X = stimuli(200); y = responses(X)
k_sta = ...
k_ols = ...
k_ridge = np.linalg.solve(X.T @ X + 30 * np.eye(P), X.T @ y)
for name, k in [("STA", k_sta), ("least squares", k_ols), ("ridge (lambda 30)", k_ridge)]:
    print(f"{name:18s} relative error {rel_err(k):.2f}, width {width(k):.2f} (true width {width(k_true):.2f})")
plt.plot(k_true, "k--", label="true"); plt.plot(k_sta, label="STA"); plt.plot(k_ols, label="least squares"); plt.plot(k_ridge, label="ridge"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_stim():
    "neighbouring pixels are correlated at about 0.8"
    c = np.corrcoef(X[:, 10], X[:, 11])[0, 1]; assert 0.7 < c < 0.9
def test_estimates():
    "k_sta is X'y/n rescaled by the mean pixel variance, and k_ols solves the normal equations"
    assert np.allclose(k_sta, X.T @ y / len(y) / X.var(0).mean()) and np.allclose(X.T @ X @ k_ols, X.T @ y)
def test_errors():
    "the STA is wider than the true field (biased), and least squares is noisier than ridge"
    assert width(k_sta) > width(k_true) and rel_err(k_ols) > rel_err(k_ridge)

test_stim()
test_estimates()
test_errors()
print("All checks passed")

### Exercise: Ridge against smoothness, both cross-validated

Build D. Write cv_choose(X, y, R), which picks lambda for the penalty lambda * k' R k by five-fold cross-validation over lams. Fit least squares, ridge (R = I) and smoothness (R = D'D) and compare their field errors and their R² on 5,000 new trials.

In [ ]:
D = ...
lams = np.logspace(-1, 4, 21)

def fit(X, y, R, lam):
    return np.linalg.solve(X.T @ X + lam * R, X.T @ y)

def cv_choose(X, y, R, K=5):
    folds = np.array_split(np.arange(len(y)), K)
    errs = []
    for lam in lams:
        e = 0.0
        for te in folds:
            tr = np.setdiff1d(np.arange(len(y)), te)
            e += ...
        errs.append(e / len(y))
    return lams[int(np.argmin(errs))], np.array(errs)

X_new = stimuli(5000); y_new = responses(X_new)
r2_of = lambda k: 1 - np.mean((y_new - X_new @ k) ** 2) / y_new.var()
lam_r, _ = cv_choose(X, y, np.eye(P)); lam_s, _ = cv_choose(X, y, D.T @ D)
ests = {"least squares": k_ols, "ridge": fit(X, y, np.eye(P), lam_r), "smooth": fit(X, y, D.T @ D, lam_s)}
err = {k: rel_err(v) for k, v in ests.items()}; r2 = {k: r2_of(v) for k, v in ests.items()}
print(f"chosen penalties: ridge {lam_r:.1f}, smoothness {lam_s:.1f}")
for name in ests:
    print(f"{name:14s} field error {err[name]:.2f}, held-out R² {r2[name]:.3f}")
print(f"the true field's R²: {r2_of(k_true):.3f}")

In [ ]:
import numpy as np
def test_D():
    "D is the (P - 2) x P second-difference matrix"
    assert D.shape == (P - 2, P) and np.allclose(D @ np.ones(P), 0) and np.allclose(D @ pix, 0)
def test_cv():
    "cv_choose returns the penalty with the lowest five-fold error"
    lam, errs = cv_choose(X, y, np.eye(P))
    assert lam == lams[np.argmin(errs)]
def test_better():
    "the smoothness penalty gives the lowest field error, and both penalties beat least squares on held-out R²"
    assert err["smooth"] < err["ridge"] < err["least squares"] and min(r2["smooth"], r2["ridge"]) > r2["least squares"]

test_D()
test_cv()
test_better()
print("All checks passed")

### Exercise: The value of a prior shrinks with data

For each number of trials, draw new stimuli and responses, fit least squares and the cross-validated smoothness estimate, and record their held-out R². Store the difference (smooth minus least squares) in gain.

In [ ]:
ns = [60, 120, 250, 500, 1000]
gain = np.zeros(len(ns))
for i, n_ in enumerate(ns):
    Xn = stimuli(n_); yn = responses(Xn)
    k_o = np.linalg.solve(Xn.T @ Xn, Xn.T @ yn)
    lam_n, _ = cv_choose(Xn, yn, D.T @ D)
    k_s = fit(Xn, yn, D.T @ D, lam_n)
    gain[i] = ...
    print(f"{n_:5d} trials: least squares R² {r2_of(k_o):.3f}, smoothness R² {r2_of(k_s):.3f}")

In [ ]:
import numpy as np
def test_gap():
    "the smoothness penalty's advantage over least squares is larger at 60 trials than at 1,000"
    assert gain[0] > gain[-1] > -0.01

test_gap()
print("All checks passed")